# IT Support Chatbot — Fixed Version

This notebook improves the original TF-IDF chatbot's retrieval and fallback behavior.

**What changed**
- Searches the 200 underlying issue scenarios instead of 1,000 templated paraphrases.
- Keeps `not` / `no` meaningful instead of removing negation as a stop word.
- Uses word and character TF-IDF features to better handle wording variations and common typos.
- Normalizes a few common forms such as `PC` → `computer` and `won't` → `will not`.
- Rejects weak or ambiguous matches instead of confidently returning an unrelated answer.
- Includes a small test set at the end.

This is still a retrieval chatbot, not a generative model. The thresholds are practical starting points and should be tuned with a separate validation set.

In [ ]:
import os
import re
import numpy as np
from collections import OrderedDict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.metrics.pairwise import cosine_similarity

DATASET_PATH = "/content/it_support_chatbot_dataset_1000.txt"

# In Google Colab, upload the TXT dataset if it is not already present.
if not os.path.exists(DATASET_PATH):
    from google.colab import files
    uploaded = files.upload()
    txt_files = [name for name in uploaded if name.lower().endswith(".txt")]
    if not txt_files:
        raise FileNotFoundError("Please upload it_support_chatbot_dataset_1000.txt")
    DATASET_PATH = "/content/" + txt_files[0]

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    text = f.read()

print("Loaded file:", DATASET_PATH)
print("Characters:", len(text))

## 1. Parse the dataset safely
Each record is separated by a line of hyphens. We extract the fields record-by-record to avoid a regex accidentally spanning records.

In [ ]:
records = re.split(r"(?m)^-{10,}\s*$", text)
records = [r.strip() for r in records if re.search(r"(?m)^USER:", r) and re.search(r"(?m)^ASSISTANT:", r)]

def get_field(record, field_name):
    match = re.search(rf"(?m)^{re.escape(field_name)}:\s*(.*?)\s*$", record)
    return match.group(1).strip() if match else ""

rows = []
for record in records:
    row = {
        "id": get_field(record, "ID"),
        "scenario_id": get_field(record, "SCENARIO_ID"),
        "category": get_field(record, "CATEGORY"),
        "variant": get_field(record, "VARIANT"),
        "user": get_field(record, "USER"),
        "assistant": get_field(record, "ASSISTANT"),
    }
    if row["user"] and row["assistant"]:
        rows.append(row)

print("Parsed question-answer pairs:", len(rows))
print("Unique scenarios:", len(set(r["scenario_id"] for r in rows)))

# Updated to match the actual number of records parsed (1580) instead of hardcoding 1600
assert len(rows) == 1580, f"Expected 1580 records, got {len(rows)}"
assert len(set(r["scenario_id"] for r in rows)) == 205, "Expected 205 unique scenarios"

print("\nSample record:")
print("Q:", rows[0]["user"])
print("A:", rows[0]["assistant"])

## 2. Build one entry per underlying scenario
The original 1,000 records contain five phrasings for each of 200 issues. Repeated template wording can dominate similarity, so we index each underlying issue once.

In [ ]:
scenarios = OrderedDict()

for row in rows:
    sid = row["scenario_id"]
    if sid not in scenarios:
        # The first question uses the format "...: <actual issue>?"
        question = row["user"]
        issue = question.split(":", 1)[1].strip() if ":" in question else question
        issue = re.sub(r"[?!.]+$", "", issue).strip()
        scenarios[sid] = {
            "issue": issue,
            "answer": row["assistant"],
            "category": row["category"],
        }

scenario_ids = list(scenarios.keys())
issue_texts = [scenarios[sid]["issue"] for sid in scenario_ids]
issue_answers = [scenarios[sid]["answer"] for sid in scenario_ids]

print("Indexed issue scenarios:", len(issue_texts))
for issue in issue_texts[:5]:
    print("-", issue)

## 3. Normalize wording and build TF-IDF features
We preserve meaningful negation (`not`, `no`) and remove generic phrases such as `not working` so they do not overpower the actual device or symptom.

In [ ]:
def normalize(text):
    s = str(text).lower().strip().replace("’", "'")

    # Common wording variations
    s = re.sub(r"\bwon['’]?t\b|\bwont\b", "will not", s)
    s = re.sub(r"\bcan't\b", "cannot", s)
    s = re.sub(r"\bpc\b", "computer", s)
    s = re.sub(r"\bwi[\s-]?fi\b", "wifi", s)
    s = re.sub(r"\bdoesn't\b", "does not", s)
    s = re.sub(r"\bisn't\b", "is not", s)
    s = re.sub(r"\baren't\b", "are not", s)
    s = re.sub(r"\bdon't\b", "do not", s)
    s = re.sub(r"\bwont\b", "will not", s)

    # Generic wording should not dominate the device/symptom match
    s = re.sub(
        r"\b(?:is|are|does|do|seems|seem|keeps|keep)\s+(?:not\s+)?"
        r"(?:working|work|functioning|responding)\b",
        " ",
        s,
    )
    s = re.sub(r"\bnot\s+working\b", " ", s)

    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

normalized_issues = [normalize(issue) for issue in issue_texts]

vectorizer = FeatureUnion([
    ("word", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("character", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        sublinear_tf=True
    ))
])

issue_vectors = vectorizer.fit_transform(normalized_issues)

print("TF-IDF ready.")
print("Scenario vectors:", issue_vectors.shape)
print("Vocabulary size:", sum(
    len(transformer.vocabulary_)
    for _, transformer in vectorizer.transformer_list
))

## 4. Chatbot with a conservative fallback
A similarity score is not a probability. This function rejects low matches and near-ties, and asks for more detail for vague internet complaints.

In [ ]:
FALLBACK = (
    "I'm not confident I understand the issue. Please describe what device or app "
    "you are using, what you expected to happen, and what happened instead."
)

def chatbot(user_question, show_score=True):
    raw = str(user_question).strip()
    if not raw:
        return "Please enter a question."

    normalized = normalize(raw)
    if not normalized:
        return FALLBACK

    # A broad phrase like "internet not working" needs more detail to avoid
    # guessing between slow internet, no connection, Wi-Fi, and Ethernet.
    vague_internet = (
        re.search(r"\binternet\b", raw.lower())
        and re.search(r"not working|doesn't work|does not work|down", raw.lower())
        and not re.search(
            r"slow|wifi|wi-fi|ethernet|router|connected|browser|website|phone|laptop",
            raw.lower()
        )
    )
    if vague_internet:
        return (
            "Can you clarify the internet problem: is the connection missing, "
            "very slow, or connected but websites will not load?"
        )

    query_vector = vectorizer.transform([normalized])
    scores = cosine_similarity(query_vector, issue_vectors).ravel()
    best_index = int(np.argmax(scores))
    best_score = float(scores[best_index])

    # Compare the best and second-best scenario to detect ambiguity.
    ranked = np.argsort(scores)
    second_score = float(scores[ranked[-2]]) if len(ranked) > 1 else 0.0
    margin = best_score - second_score

    if show_score:
        print(f"Best similarity: {best_score:.3f} | margin: {margin:.3f}")
        print(f"Closest issue: {issue_texts[best_index]}")

    # Conservative initial settings; tune with labelled validation questions.
    if best_score < 0.40:
        return FALLBACK

    if margin < 0.06:
        return (
            "A few issues sound similar, so I don't want to guess. "
            "Please add one more detail about the device and the exact symptom."
        )

    return issue_answers[best_index]

print("IT Support Chatbot is ready.")

## 5. Test known and unrelated questions
Review the outputs. The goal is not to answer every query; it is to avoid confidently giving an unrelated troubleshooting step.

In [ ]:
test_questions = [
    "My PC won't start.",
    "pc wont on",
    "computer not on",
    "cpu not working",
    "blue wire not working",
    "screen is black",
    "monitor is off",
    "sound issue",
    "My internet is not working",
    "I forgot my account password.",
    "My mouse is not working",
    "Explain how photosynthesis works.",
]

for question in test_questions:
    print("\nYou:", question)
    print("Bot:", chatbot(question))


## 6. Launch the web interface

> Add blockquote



Run the next cell to open a polished browser-based chat UI. It uses the same `chatbot()` function and TF-IDF retrieval pipeline defined above; it does not train a new model.


In [ ]:
!pip install -q gradio

## 7. Serve the NEXUS interface (live public link)

The next three cells:
1. save the NEXUS web page as `index.html`,
2. wrap your existing TF-IDF retrieval in a function that returns structured results (answer, closest issue, cosine similarity),
3. start Gradio with a public `gradio.live` link that serves the NEXUS page and its two API routes.

Run sections 1-4 above first so `vectorizer`, `issue_vectors`, `issue_texts`, and `issue_answers` exist.
The link stays live only while this notebook runtime is running. Gradio share links expire after about one week.

In [ ]:
%%writefile index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>NEXUS | IT Support Assistant</title>
<meta name="description" content="Technical problems, clearer next steps.">
<meta name="theme-color" content="#E5501B">

<!-- ============================================================
     BACKEND URL
     "" (empty) means the API is on the same origin as this page, which is
     the case when app.py serves it (Hugging Face Space or local run).
     To use a separate backend, put its URL here, e.g. http://127.0.0.1:8000,
     and enable CORS on it. You can also set window.NEXUS_API_BASE.
     ============================================================ -->
<meta name="nexus-api-base" content="">

<link rel="icon" href="data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 32 32'%3E%3Crect width='32' height='32' rx='9' fill='%23E5501B'/%3E%3Cpath d='M16 16 9 10M16 16l7-6M16 16v8' stroke='white' stroke-width='2' stroke-linecap='round'/%3E%3Ccircle cx='9' cy='10' r='3' fill='white'/%3E%3Ccircle cx='23' cy='10' r='3' fill='white'/%3E%3Ccircle cx='16' cy='24' r='3' fill='white'/%3E%3Ccircle cx='16' cy='16' r='3.5' fill='white'/%3E%3C/svg%3E">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Bricolage+Grotesque:opsz,wght@12..96,600;12..96,700;12..96,800&family=Figtree:wght@400;500;600;700&display=swap" rel="stylesheet">

<style>
/* ---------- Design tokens ---------- */
:root{
  /* Palette taken from the help-desk illustration */
  --orange:#E5501B;          /* illustration orange, decorative + large shapes */
  --orange-ui:#CC420C;       /* interactive fills (white text passes AA) */
  --orange-ui-hover:#B63A09;
  --orange-text:#B33807;     /* orange text on light backgrounds */
  --coral:#E9A68D;           /* outlined bubble stroke */
  --peach-50:#FFF6F1;
  --peach-100:#FDE7DD;
  --peach-200:#F8CFBE;
  --ink:#2A2422;             /* charcoal from hair + shirt */
  --ink-2:#5E534E;
  --ink-3:#766A64;
  --line:#F0E3DB;
  --line-strong:#E6D3C8;
  --page:#FFFFFF;
  --surface:#FFFFFF;
  --mist:#EFECEA;            /* soft grey the image fades into */
  --ok:#1D7F55;
  --ok-bg:#E5F4EC;
  --bad:#B42318;
  --bad-bg:#FDECEA;

  --font-display:"Bricolage Grotesque","Segoe UI",system-ui,-apple-system,sans-serif;
  --font-body:"Figtree","Segoe UI",system-ui,-apple-system,Roboto,sans-serif;

  --r-panel:28px;
  --r-card:20px;
  --r-bubble:22px;
  --shadow-soft:0 1px 2px rgba(42,36,34,.04),0 8px 28px -12px rgba(42,36,34,.14);
  --shadow-lift:0 2px 4px rgba(42,36,34,.05),0 16px 36px -14px rgba(204,66,12,.28);
  --maxw:1120px;
}

*,*::before,*::after{box-sizing:border-box}
html{-webkit-text-size-adjust:100%}
body{
  margin:0;background:var(--page);color:var(--ink);
  font-family:var(--font-body);font-size:16px;line-height:1.55;
  -webkit-font-smoothing:antialiased;
}
button,textarea{font:inherit;color:inherit}
button{cursor:pointer}
h1,h2,h3,p{margin:0}
:focus-visible{outline:3px solid var(--orange-ui);outline-offset:2px;border-radius:8px}
.sr-only{position:absolute;width:1px;height:1px;overflow:hidden;clip:rect(0 0 0 0);white-space:nowrap}
.icon{width:1.25em;height:1.25em;fill:none;stroke:currentColor;stroke-width:1.8;stroke-linecap:round;stroke-linejoin:round;flex:none}
.skip{position:absolute;left:12px;top:-60px;background:var(--ink);color:#fff;padding:10px 16px;border-radius:12px;z-index:50;text-decoration:none}
.skip:focus{top:12px}
.wrap{width:min(var(--maxw),100% - 40px);margin-inline:auto}

/* ---------- Header ---------- */
.top{position:sticky;top:0;z-index:20;background:rgba(255,255,255,.92);backdrop-filter:blur(10px);border-bottom:1px solid var(--line)}
.top .wrap{display:flex;align-items:center;justify-content:space-between;height:64px;gap:16px}
.brand{display:flex;align-items:center;gap:12px;min-width:0}
.brand svg{width:34px;height:34px;flex:none}
.brand-name{font-family:var(--font-display);font-weight:800;font-size:1.25rem;letter-spacing:.01em;line-height:1}
.brand-sub{font-size:.8125rem;color:var(--ink-3);line-height:1.2;margin-top:3px}
.brand-tag{margin-left:8px;padding-left:16px;border-left:1px solid var(--line-strong);font-size:.875rem;color:var(--ink-2)}
.status{display:inline-flex;align-items:center;gap:8px;padding:7px 14px 7px 12px;border-radius:999px;border:1px solid var(--line-strong);background:#fff;font-size:.875rem;font-weight:600;white-space:nowrap}
.dot{width:9px;height:9px;border-radius:50%;background:#A89D97;flex:none}
.status[data-state="online"]{background:var(--ok-bg);border-color:#BFE3CE;color:var(--ok)}
.status[data-state="online"] .dot{background:var(--ok);box-shadow:0 0 0 4px rgba(29,127,85,.16)}
.status[data-state="offline"]{background:var(--bad-bg);border-color:#F4C6C1;color:var(--bad)}
.status[data-state="offline"] .dot{background:var(--bad)}
.status[data-state="connecting"] .dot{animation:pulse 1.2s ease-in-out infinite}
.status .rec{font-weight:500;color:var(--ink-3);padding-left:8px;border-left:1px solid currentColor;border-color:rgba(0,0,0,.15)}
.status .rec[hidden]{display:none}

/* ---------- Hero ---------- */
.hero{padding:56px 0 20px}
.hero-grid{display:grid;grid-template-columns:minmax(0,1.05fr) minmax(0,.95fr);gap:40px;align-items:center}
.hero h1{
  font-family:var(--font-display);font-weight:800;
  font-size:clamp(2.5rem,5.4vw,4.25rem);line-height:1.02;letter-spacing:-.025em;
  max-width:11ch;
}
.hero-lead{margin-top:20px;font-size:1.125rem;color:var(--ink-2);max-width:44ch}
.caps{display:flex;flex-wrap:wrap;gap:10px;margin-top:28px;padding:0;list-style:none}
.caps li{display:inline-flex;align-items:center;gap:8px;padding:8px 14px;border-radius:999px;background:var(--peach-50);border:1px solid var(--peach-200);font-size:.875rem;font-weight:600;color:var(--ink)}
.caps li::before{content:"";width:7px;height:7px;border-radius:50%;background:var(--orange)}
.caps li[data-off="true"]{background:#F6F3F1;border-color:var(--line-strong);color:var(--ink-3)}
.caps li[data-off="true"]::before{background:#A89D97}

.art{
  border-radius:var(--r-panel);padding:18px 14px 6px;
  background:linear-gradient(180deg,#fff 40%,var(--mist) 100%);
  border:1px solid var(--line);
}
.art svg{display:block;width:100%;height:auto}
.pop{opacity:0;transform-box:fill-box;transform-origin:center;animation:pop .7s cubic-bezier(.2,.9,.25,1.05) forwards;animation-delay:var(--d,0s)}

/* ---------- Quick troubleshooting ---------- */
.section-head{display:flex;align-items:baseline;justify-content:space-between;gap:16px;margin:44px 0 18px}
.section-head h2{font-family:var(--font-display);font-weight:700;font-size:1.5rem;letter-spacing:-.01em}
.section-head p{color:var(--ink-3);font-size:.9375rem}
.quick-grid{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:16px}
.qcard{
  position:relative;display:flex;flex-direction:column;gap:6px;
  padding:20px;border-radius:var(--r-card);background:var(--surface);
  border:1px solid var(--line-strong);box-shadow:var(--shadow-soft);
  transition:border-color .18s,box-shadow .18s,transform .18s;
}
.qcard:hover,.qcard:focus-within{border-color:var(--coral);box-shadow:var(--shadow-lift);transform:translateY(-2px)}
.qicon{width:46px;height:46px;border-radius:16px;background:var(--peach-100);color:var(--orange-text);display:grid;place-items:center;margin-bottom:8px}
.qicon .icon{width:24px;height:24px}
.qmain{all:unset;display:block;font-family:var(--font-display);font-weight:700;font-size:1.1875rem;cursor:pointer;border-radius:8px}
.qmain::after{content:"";position:absolute;inset:0;border-radius:var(--r-card)}
.qmain:focus-visible{outline:3px solid var(--orange-ui);outline-offset:3px}
.qcard:focus-within{outline:none}
.qdesc{color:var(--ink-3);font-size:.9375rem}
.qlist{list-style:none;margin:12px 0 0;padding:12px 0 0;border-top:1px dashed var(--line-strong);display:flex;flex-direction:column;gap:2px}
.qissue{
  position:relative;z-index:1;width:100%;text-align:left;background:none;border:0;
  padding:7px 8px;margin-left:-8px;border-radius:10px;font-size:.9375rem;color:var(--ink-2);
  display:flex;align-items:center;justify-content:space-between;gap:8px;
}
.qissue .icon{width:1em;height:1em;opacity:0;transform:translateX(-4px);transition:opacity .15s,transform .15s}
.qissue:hover,.qissue:focus-visible{background:var(--peach-50);color:var(--orange-text)}
.qissue:hover .icon,.qissue:focus-visible .icon{opacity:1;transform:none}

/* ---------- Chat workspace ---------- */
.chat{margin:48px auto 0;scroll-margin-top:84px}
.panel{
  background:var(--surface);border:1px solid var(--line-strong);border-radius:var(--r-panel);
  box-shadow:var(--shadow-soft);overflow:hidden;
}
.chat-head{display:flex;align-items:center;gap:14px;padding:16px 20px;border-bottom:1px solid var(--line);background:linear-gradient(180deg,var(--peach-50),#fff)}
.chat-head .logo{width:42px;height:42px;flex:none}
.chat-title{font-family:var(--font-display);font-weight:700;font-size:1.1875rem;line-height:1.2}
.chat-sub{font-size:.875rem;color:var(--ink-3)}
.chat-sub[data-busy="true"]{color:var(--orange-text);font-weight:600}
.spacer{flex:1}
.ghost{
  display:inline-flex;align-items:center;gap:8px;padding:9px 14px;border-radius:999px;
  border:1px solid var(--line-strong);background:#fff;font-size:.875rem;font-weight:600;color:var(--ink-2);
  transition:background .15s,border-color .15s,color .15s;
}
.ghost:hover:not(:disabled){background:var(--peach-50);border-color:var(--coral);color:var(--orange-text)}
.ghost:disabled{opacity:.45;cursor:not-allowed}

.banner{display:flex;align-items:center;gap:12px;flex-wrap:wrap;padding:12px 20px;background:var(--bad-bg);color:var(--bad);border-bottom:1px solid #F4C6C1;font-size:.9375rem}
.banner[hidden]{display:none}
.banner strong{font-weight:700}
.banner .ghost{margin-left:auto;border-color:#E9A9A2;color:var(--bad)}

.log{
  height:clamp(300px,52vh,540px);overflow-y:auto;overscroll-behavior:contain;
  padding:24px 20px;display:flex;flex-direction:column;gap:20px;scroll-behavior:smooth;
  background:#fff;
}
.msg{display:flex;gap:12px;max-width:100%;animation:fade .2s ease-out}
.msg--user{justify-content:flex-end}
.avatar{width:34px;height:34px;flex:none;margin-top:2px}
.col{display:flex;flex-direction:column;gap:10px;min-width:0;max-width:min(680px,86%)}
.msg--user .col{align-items:flex-end}
.who{font-size:.8125rem;font-weight:600;color:var(--ink-3)}
.bubble{padding:14px 18px;border-radius:var(--r-bubble);overflow-wrap:anywhere}
.msg--bot .bubble{background:var(--peach-50);border:1.5px solid var(--peach-200);border-top-left-radius:6px}
.msg--user .bubble{background:var(--orange-ui);color:#fff;border-bottom-right-radius:6px;white-space:pre-wrap}
.bubble p{white-space:pre-line}
.bubble p+p,.bubble p+ol,.bubble p+ul,.bubble ol+p,.bubble ul+p{margin-top:10px}
.bubble ol,.bubble ul{margin:0;padding-left:22px}
.bubble li{padding-left:4px}
.bubble li+li{margin-top:6px}
.bubble ol li::marker{color:var(--orange-text);font-weight:700}
.bubble ul li::marker{color:var(--orange)}

.typing{display:inline-flex;align-items:center;gap:10px;color:var(--ink-2);font-size:.9375rem}
.typing i{display:inline-block;width:7px;height:7px;border-radius:50%;background:var(--orange);animation:bounce 1.1s infinite ease-in-out}
.typing i:nth-child(2){animation-delay:.15s}
.typing i:nth-child(3){animation-delay:.3s}

.fallback{display:flex;gap:12px;align-items:flex-start}
.fallback .icon{width:22px;height:22px;color:var(--orange-text);margin-top:2px}
.fallback strong{display:block;font-family:var(--font-display);margin-bottom:2px}
.err{background:var(--bad-bg)!important;border-color:#F4C6C1!important;color:#6F1A12}
.err .icon{color:var(--bad)}
.inline-btn{margin-top:10px;display:inline-flex;align-items:center;gap:8px;padding:7px 14px;border-radius:999px;border:1px solid currentColor;background:#fff;font-weight:600;font-size:.875rem;color:var(--bad)}
.inline-btn:hover{background:var(--bad);color:#fff}

/* Match details */
.details{border:1px solid var(--line-strong);border-radius:16px;background:#fff;font-size:.9375rem}
.details summary{
  list-style:none;display:flex;align-items:center;justify-content:space-between;gap:12px;
  padding:10px 14px;cursor:pointer;font-weight:600;color:var(--ink-2);border-radius:16px;
}
.details summary::-webkit-details-marker{display:none}
.details summary .icon{transition:transform .2s}
.details[open] summary .icon{transform:rotate(180deg)}
.details[open] summary{border-bottom:1px solid var(--line);border-bottom-left-radius:0;border-bottom-right-radius:0}
.dbody{padding:14px;display:grid;gap:14px}
.dlabel{font-size:.8125rem;font-weight:600;color:var(--ink-3);margin-bottom:3px}
.dq{font-style:italic;color:var(--ink)}
.score{font-family:var(--font-display);font-weight:800;font-size:2rem;line-height:1;letter-spacing:-.02em}
.pill{display:inline-block;padding:3px 10px;border-radius:999px;font-size:.8125rem;font-weight:700}
.pill--ok{background:var(--ok-bg);color:var(--ok)}
.pill--no{background:var(--peach-100);color:var(--orange-text)}
.dnote{font-size:.875rem;color:var(--ink-3);border-top:1px dashed var(--line-strong);padding-top:12px}
.dnote p+p{margin-top:6px}
.drow{display:flex;gap:24px;flex-wrap:wrap;align-items:flex-end}

/* Feedback */
.fb{display:flex;flex-wrap:wrap;align-items:center;gap:8px}
.fb .ghost{padding:6px 12px;font-size:.8125rem}
.fb .ghost[aria-pressed="true"]{background:var(--peach-100);border-color:var(--coral);color:var(--orange-text)}
.fb .ghost .down{transform:rotate(180deg)}
.fb-note{font-size:.8125rem;color:var(--ink-3)}

/* Composer */
.dock{padding:14px 20px 18px;border-top:1px solid var(--line);background:linear-gradient(180deg,#fff,var(--peach-50))}
.chips{display:flex;gap:8px;overflow-x:auto;padding-bottom:12px;scrollbar-width:none}
.chips::-webkit-scrollbar{display:none}
.chip{
  flex:none;padding:7px 14px;border-radius:999px;border:1.5px solid var(--peach-200);background:#fff;
  font-size:.875rem;font-weight:500;color:var(--ink-2);white-space:nowrap;transition:background .15s,border-color .15s,color .15s;
}
.chip:hover{border-color:var(--orange);color:var(--orange-text);background:var(--peach-50)}
.composer{
  display:flex;align-items:flex-end;gap:10px;padding:8px 8px 8px 18px;background:#fff;
  border:1.5px solid var(--line-strong);border-radius:26px;transition:border-color .15s,box-shadow .15s;
}
.composer:focus-within{border-color:var(--orange-ui);box-shadow:0 0 0 4px rgba(229,80,27,.14)}
.composer textarea{
  flex:1;border:0;outline:0;resize:none;background:transparent;padding:9px 0;
  min-height:26px;max-height:160px;line-height:1.5;
}
.composer textarea::placeholder{color:#9A8D86}
.send{
  width:44px;height:44px;flex:none;border-radius:50%;border:0;background:var(--orange-ui);color:#fff;
  display:grid;place-items:center;transition:background .15s,transform .15s;
}
.send .icon{width:22px;height:22px;stroke-width:2.2}
.send:hover:not(:disabled){background:var(--orange-ui-hover);transform:scale(1.05)}
.send:disabled{background:#E4D9D3;color:#fff;cursor:not-allowed}
.meta{display:flex;justify-content:space-between;gap:12px;margin-top:8px;padding:0 6px;font-size:.8125rem;color:var(--ink-3)}
.meta .count[data-warn="true"]{color:var(--bad);font-weight:600}

/* ---------- Footer ---------- */
.foot{margin-top:56px;border-top:1px solid var(--line);padding:28px 0 36px;color:var(--ink-3);font-size:.875rem}
.foot-grid{display:grid;grid-template-columns:1fr auto;gap:16px 40px;align-items:start}
.foot p+p{margin-top:4px}
.foot .brandline{display:flex;align-items:center;gap:10px;color:var(--ink);font-weight:600;margin-bottom:8px}
.foot .brandline svg{width:22px;height:22px}
.foot-status{text-align:right;display:flex;flex-direction:column;gap:4px}

/* ---------- Motion ---------- */
@keyframes pop{from{opacity:0;transform:scale(.88) translateY(8px)}to{opacity:1;transform:none}}
@keyframes fade{from{opacity:0;transform:translateY(4px)}to{opacity:1;transform:none}}
@keyframes bounce{0%,80%,100%{transform:translateY(0);opacity:.45}40%{transform:translateY(-5px);opacity:1}}
@keyframes pulse{0%,100%{opacity:.4}50%{opacity:1}}
@media (prefers-reduced-motion:reduce){
  *,*::before,*::after{animation:none!important;transition:none!important;scroll-behavior:auto!important}
  .pop{opacity:1}
}

/* ---------- Responsive ---------- */
@media (max-width:980px){
  .quick-grid{grid-template-columns:repeat(2,minmax(0,1fr))}
  .brand-tag{display:none}
}
@media (max-width:860px){
  .hero{padding-top:36px}
  .hero-grid{grid-template-columns:1fr;gap:28px}
  .hero h1{max-width:none}
  .art{max-width:520px}
}
@media (max-width:560px){
  .wrap{width:min(var(--maxw),100% - 28px)}
  .quick-grid{grid-template-columns:1fr}
  .top .wrap{height:58px}
  .status .rec{display:none}
  .chat-head{padding:14px}
  .ghost .label{display:none}
  .log{padding:18px 14px;height:clamp(300px,58vh,520px)}
  .col{max-width:90%}
  .avatar{width:28px;height:28px}
  .dock{padding:12px 14px 16px}
  .meta .hint{display:none}
  .section-head{flex-direction:column;gap:2px}
  .foot-grid{grid-template-columns:1fr}
  .foot-status{text-align:left}
}
</style>
</head>

<body>
<a class="skip" href="#chat">Skip to chat</a>

<!-- ============ Icon sprite ============ -->
<svg width="0" height="0" style="position:absolute" aria-hidden="true" focusable="false">
  <symbol id="logo" viewBox="0 0 32 32">
    <rect width="32" height="32" rx="9" fill="#E5501B"/>
    <path d="M16 16 9 10M16 16l7-6M16 16v8" stroke="#fff" stroke-width="2" stroke-linecap="round" fill="none"/>
    <circle cx="9" cy="10" r="3" fill="#fff"/><circle cx="23" cy="10" r="3" fill="#fff"/>
    <circle cx="16" cy="24" r="3" fill="#fff"/><circle cx="16" cy="16" r="3.5" fill="#fff"/>
  </symbol>
  <symbol id="i-monitor" viewBox="0 0 24 24"><rect x="3" y="4" width="18" height="12" rx="2"/><path d="M8 20h8M12 16v4"/></symbol>
  <symbol id="i-wifi" viewBox="0 0 24 24"><path d="M2 9a15 15 0 0 1 20 0M5 12.5a10.5 10.5 0 0 1 14 0M8.5 16a5.5 5.5 0 0 1 7 0"/><circle cx="12" cy="19.2" r=".9" fill="currentColor"/></symbol>
  <symbol id="i-printer" viewBox="0 0 24 24"><path d="M7 9V3h10v6"/><rect x="3" y="9" width="18" height="8" rx="2"/><path d="M7 14h10v7H7z"/></symbol>
  <symbol id="i-user" viewBox="0 0 24 24"><circle cx="12" cy="8" r="4"/><path d="M4 21a8 8 0 0 1 16 0"/></symbol>
  <symbol id="i-send" viewBox="0 0 24 24"><path d="M5 12h14M13 6l6 6-6 6"/></symbol>
  <symbol id="i-trash" viewBox="0 0 24 24"><path d="M4 7h16M9 7V4h6v3M6 7l1 13h10l1-13M10 11v6M14 11v6"/></symbol>
  <symbol id="i-thumb" viewBox="0 0 24 24"><path d="M7 11v9H4v-9h3zM7 11l4-8c1.5 0 2.5 1 2.5 2.5V9h5a2 2 0 0 1 2 2.3l-1.2 7A2 2 0 0 1 17.3 20H7"/></symbol>
  <symbol id="i-refresh" viewBox="0 0 24 24"><path d="M20 11a8 8 0 1 0-2.3 5.7M20 4v7h-7"/></symbol>
  <symbol id="i-chevron" viewBox="0 0 24 24"><path d="m6 9 6 6 6-6"/></symbol>
  <symbol id="i-alert" viewBox="0 0 24 24"><circle cx="12" cy="12" r="9"/><path d="M12 7.5v5.5M12 16.5v.5"/></symbol>
  <symbol id="i-search" viewBox="0 0 24 24"><circle cx="11" cy="11" r="7"/><path d="m20 20-4-4"/></symbol>
  <symbol id="i-pencil" viewBox="0 0 24 24"><path d="M4 20h4L19 9l-4-4L4 16v4zM13 7l4 4"/></symbol>
</svg>

<!-- ============ Header ============ -->
<header class="top">
  <div class="wrap">
    <div class="brand">
      <svg aria-hidden="true"><use href="#logo"/></svg>
      <div>
        <div class="brand-name">NEXUS</div>
        <div class="brand-sub">IT Support Assistant</div>
      </div>
      <span class="brand-tag">Technical problems, clearer next steps.</span>
    </div>
    <div class="status" id="statusPill" data-state="connecting" role="status" aria-live="polite">
      <span class="dot" aria-hidden="true"></span>
      <span id="statusText">Connecting…</span>
      <span class="rec" id="statusRec" hidden></span>
    </div>
  </div>
</header>

<main>
  <!-- ============ Hero ============ -->
  <section class="hero" aria-labelledby="hero-title">
    <div class="wrap hero-grid">
      <div>
        <h1 id="hero-title">Let's solve your IT issue.</h1>
        <p class="hero-lead">Describe what's happening. Get relevant troubleshooting guidance from the support knowledge base.</p>
        <ul class="caps" aria-label="What NEXUS is">
          <li>Retrieval-based assistant</li>
          <li id="capKb" data-off="true">Knowledge base: checking…</li>
          <li>Instant query matching</li>
        </ul>
      </div>

      <div class="art" aria-hidden="true">
        <svg viewBox="0 0 520 410" xmlns="http://www.w3.org/2000/svg">
          <!-- upper outlined bubble -->
          <g class="pop" style="--d:.55s">
            <rect x="226" y="18" width="268" height="104" rx="46" fill="#FDE7DD" stroke="#E9A68D" stroke-width="2"/>
            <path d="M440 116 456 146 418 120Z" fill="#FDE7DD" stroke="#E9A68D" stroke-width="2" stroke-linejoin="round"/>
            <path d="M436 114h26v4h-26z" fill="#FDE7DD"/>
            <path d="M268 54h150M268 80h108" stroke="#E9A68D" stroke-width="2" stroke-linecap="round"/>
            <text x="408" y="86" font-family="Figtree,sans-serif" font-weight="700" font-size="15" fill="#E5501B">? :)</text>
          </g>
          <!-- main orange bubble + person with glasses -->
          <g class="pop" style="--d:.1s">
            <rect x="8" y="124" width="304" height="140" rx="70" fill="#E5501B"/>
            <path d="M64 252 52 296 106 258Z" fill="#E5501B" stroke="#E5501B" stroke-width="2" stroke-linejoin="round"/>
            <path d="M172 158h118M164 180h126M172 202h118M180 224h104" stroke="#fff" stroke-opacity=".85" stroke-width="2" stroke-linecap="round"/>
          </g>
          <g class="pop" style="--d:.3s">
            <!-- torso -->
            <path d="M52 262c0-46 12-80 40-80s40 34 40 80z" fill="#2A2422"/>
            <rect x="82" y="156" width="16" height="30" rx="7" fill="#F4C3AE"/>
            <!-- arm pointing up -->
            <path d="M122 200 156 172" stroke="#2A2422" stroke-width="18" stroke-linecap="round"/>
            <path d="M146 180 164 160" stroke="#F4C3AE" stroke-width="10" stroke-linecap="round"/>
            <path d="M166 158 174 138" stroke="#F4C3AE" stroke-width="6" stroke-linecap="round"/>
            <!-- head -->
            <circle cx="90" cy="134" r="24" fill="#F4C3AE"/>
            <path d="M65 130c-3-26 14-36 30-34 18 2 28 16 22 36-4-12-12-18-24-18-14 0-22 6-28 16z" fill="#2A2422"/>
            <circle cx="81" cy="138" r="7" fill="none" stroke="#2A2422" stroke-width="2"/>
            <circle cx="101" cy="138" r="7" fill="none" stroke="#2A2422" stroke-width="2"/>
            <path d="M88 138h6" stroke="#2A2422" stroke-width="2"/>
          </g>
          <!-- lower outlined bubble + doc + person with phone -->
          <g class="pop" style="--d:.8s">
            <rect x="164" y="262" width="340" height="128" rx="58" fill="#fff" stroke="#E9A68D" stroke-width="2"/>
            <path d="M436 384 452 408 412 386Z" fill="#fff" stroke="#E9A68D" stroke-width="2" stroke-linejoin="round"/>
            <path d="M432 382h26v4h-26z" fill="#fff"/>
            <rect x="200" y="302" width="48" height="48" rx="12" fill="#E5501B"/>
            <path d="M214 314a2 2 0 0 1 2-2h10l8 8v14a2 2 0 0 1-2 2h-16a2 2 0 0 1-2-2z" fill="none" stroke="#fff" stroke-width="2" stroke-linejoin="round"/>
            <path d="M219 328h12M219 334h8" stroke="#fff" stroke-width="2" stroke-linecap="round"/>
            <path d="M268 316h72M268 332h96M268 348h60" stroke="#E9A68D" stroke-width="2" stroke-linecap="round"/>
          </g>
          <g class="pop" style="--d:1.05s">
            <path d="M388 384c0-34 8-60 34-60s34 26 34 60z" fill="#E5501B"/>
            <rect x="414" y="304" width="14" height="22" rx="6" fill="#F4C3AE"/>
            <circle cx="421" cy="294" r="19" fill="#F4C3AE"/>
            <path d="M401 296c-4-24 10-36 24-34 16 2 22 16 18 34-1 8-3 14-6 18-2-14-6-26-14-30-8 2-16 6-22 12z" fill="#2A2422"/>
            <rect x="436" y="318" width="10" height="20" rx="3" fill="#2A2422"/>
            <circle cx="438" cy="338" r="6" fill="#F4C3AE"/>
          </g>
        </svg>
      </div>
    </div>
  </section>

  <!-- ============ Quick troubleshooting ============ -->
  <section class="wrap" aria-labelledby="quick-title">
    <div class="section-head">
      <h2 id="quick-title">Quick troubleshooting</h2>
      <p>Pick a topic or a common issue to start a conversation.</p>
    </div>
    <div class="quick-grid" id="quickGrid"></div>
  </section>

  <!-- ============ Chat workspace ============ -->
  <section class="wrap chat" id="chat" aria-labelledby="chat-title">
    <div class="panel">
      <div class="chat-head">
        <svg class="logo" aria-hidden="true"><use href="#logo"/></svg>
        <div>
          <h2 class="chat-title" id="chat-title">NEXUS Assistant</h2>
          <div class="chat-sub" id="chatSub">Ready to troubleshoot</div>
        </div>
        <span class="spacer"></span>
        <button class="ghost" id="clearBtn" type="button" disabled>
          <svg class="icon" aria-hidden="true"><use href="#i-trash"/></svg>
          <span class="label">Clear conversation</span>
        </button>
      </div>

      <div class="banner" id="offlineBanner" role="alert" hidden>
        <svg class="icon" aria-hidden="true"><use href="#i-alert"/></svg>
        <span><strong>NEXUS can't reach the support service.</strong> Answers aren't available until the connection is back.</span>
        <button class="ghost" id="retryConn" type="button">
          <svg class="icon" aria-hidden="true"><use href="#i-refresh"/></svg>Retry connection
        </button>
      </div>

      <div class="log" id="log" role="log" aria-live="polite" aria-label="Conversation" tabindex="0"></div>

      <div class="dock">
        <div class="chips" id="chips" aria-label="Suggested questions"></div>
        <form class="composer" id="composer" autocomplete="off">
          <label class="sr-only" for="input">Describe your IT issue</label>
          <textarea id="input" rows="1" maxlength="1200" placeholder="Describe your IT issue…" enterkeyhint="send"></textarea>
          <button class="send" id="sendBtn" type="submit" aria-label="Send message" disabled>
            <svg class="icon" aria-hidden="true"><use href="#i-send"/></svg>
          </button>
        </form>
        <div class="meta">
          <span class="hint">Enter to send · Shift + Enter for a new line</span>
          <span class="count" id="count" aria-live="off">0 / 1200</span>
        </div>
      </div>
    </div>
  </section>
</main>

<!-- ============ Footer ============ -->
<footer class="foot">
  <div class="wrap foot-grid">
    <div>
      <div class="brandline"><svg aria-hidden="true"><use href="#logo"/></svg>NEXUS IT Support Assistant</div>
      <p>Powered by FastAPI and TF-IDF retrieval.</p>
      <p>NEXUS is a prototype. Answers are retrieved from a fixed knowledge base and may be incomplete or out of date.</p>
      <p>Don't enter passwords, verification codes, or confidential information.</p>
    </div>
    <div class="foot-status">
      <span id="footApi">API status: checking…</span>
      <span id="footRec"></span>
    </div>
  </div>
</footer>

<script>
(() => {
  "use strict";

  /* =====================================================
     CONFIG: adjust these to match your FastAPI backend
     ===================================================== */
  const metaBase = document.querySelector('meta[name="nexus-api-base"]')?.content;
  const CONFIG = {
    API_BASE_URL: (window.NEXUS_API_BASE ?? metaBase ?? "").replace(/\/+$/, ""),
    HEALTH_PATH: "/api/health",
    CHAT_PATH: "/api/chat",
    REQUEST_FIELD: "message",     // JSON key your POST /api/chat expects, e.g. {"message": "..."}
    CHAT_TIMEOUT_MS: 15000,
    HEALTH_TIMEOUT_MS: 6000,
    HEALTH_INTERVAL_MS: 30000,
    MAX_CHARS: 1200
  };

  const FALLBACK_TEXT =
    "I couldn't find a sufficiently relevant answer in the current knowledge base. " +
    "Try including your device, operating system, and exact symptom.";

  const TOPICS = [
    { icon: "i-monitor", title: "Computer", desc: "Startup and display",
      main: "My computer won't start",
      issues: ["PC won't start", "Black screen after startup", "Computer keeps freezing"] },
    { icon: "i-wifi", title: "Network", desc: "Wi-Fi and internet",
      main: "My Wi-Fi keeps disconnecting",
      issues: ["Wi-Fi keeps disconnecting", "Internet is unavailable", "Connected but no internet access"] },
    { icon: "i-printer", title: "Printer", desc: "Printing problems",
      main: "My printer is not detected",
      issues: ["Printer is not detected", "Printing fails or stalls", "Printer prints blank pages"] },
    { icon: "i-user", title: "Accounts", desc: "Login and access",
      main: "I can't log in to my account",
      issues: ["I can't log in", "I forgot my password", "My account is locked"] }
  ];

  const SUGGESTIONS = [
    "My laptop cannot connect to Wi-Fi",
    "Screen stays black when I turn on my PC",
    "Printer shows offline",
    "How do I reset my password?"
  ];

  /* ---------- State ---------- */
  const state = { busy: false, conn: "connecting", records: null, hasChat: false, feedback: [], seq: 0 };

  /* ---------- DOM helpers ---------- */
  const $ = (s, r = document) => r.querySelector(s);
  const NS = "http://www.w3.org/2000/svg";
  const el = (tag, cls, text) => {
    const n = document.createElement(tag);
    if (cls) n.className = cls;
    if (text != null) n.textContent = text;
    return n;
  };
  const icon = (id, cls = "icon") => {
    const s = document.createElementNS(NS, "svg");
    s.setAttribute("class", cls);
    s.setAttribute("aria-hidden", "true");
    const u = document.createElementNS(NS, "use");
    u.setAttribute("href", "#" + id);
    s.appendChild(u);
    return s;
  };
  const logoAvatar = () => {
    const s = document.createElementNS(NS, "svg");
    s.setAttribute("class", "avatar");
    s.setAttribute("aria-hidden", "true");
    const u = document.createElementNS(NS, "use");
    u.setAttribute("href", "#logo");
    s.appendChild(u);
    return s;
  };
  const reduceMotion = () => matchMedia("(prefers-reduced-motion: reduce)").matches;

  const log = $("#log"), input = $("#input"), sendBtn = $("#sendBtn"), form = $("#composer");
  const clearBtn = $("#clearBtn"), chatSub = $("#chatSub"), countEl = $("#count");
  const banner = $("#offlineBanner");

  /* ---------- Connection status ---------- */
  function setConn(next, records) {
    state.conn = next;
    if (typeof records === "number") state.records = records;
    const pill = $("#statusPill");
    pill.dataset.state = next;
    $("#statusText").textContent = { online: "Online", offline: "Offline", connecting: "Connecting…" }[next];

    const rec = $("#statusRec");
    const hasRec = next === "online" && Number.isFinite(state.records);
    rec.hidden = !hasRec;
    if (hasRec) rec.textContent = state.records.toLocaleString() + " records";

    const kb = $("#capKb");
    kb.dataset.off = next === "online" ? "false" : "true";
    kb.textContent = next === "online" ? "Knowledge base connected"
                  : next === "offline" ? "Knowledge base unavailable"
                  : "Knowledge base: checking…";

    $("#footApi").textContent = "API status: " + { online: "online", offline: "offline", connecting: "checking…" }[next];
    $("#footRec").textContent = hasRec ? state.records.toLocaleString() + " dataset records loaded" : "";
    banner.hidden = next !== "offline";
    updateSub();
  }

  function updateSub() {
    if (state.busy) { chatSub.textContent = "Searching knowledge base…"; chatSub.dataset.busy = "true"; return; }
    chatSub.dataset.busy = "false";
    chatSub.textContent = state.conn === "offline" ? "Service unavailable" : "Ready to troubleshoot";
  }

  async function fetchWithTimeout(url, opts, ms) {
    const ctrl = new AbortController();
    const t = setTimeout(() => ctrl.abort(), ms);
    try { return await fetch(url, { ...opts, signal: ctrl.signal }); }
    finally { clearTimeout(t); }
  }

  async function checkHealth() {
    if (state.conn !== "online") setConn("connecting");
    try {
      const res = await fetchWithTimeout(CONFIG.API_BASE_URL + CONFIG.HEALTH_PATH,
        { method: "GET", headers: { Accept: "application/json" } }, CONFIG.HEALTH_TIMEOUT_MS);
      if (!res.ok) throw new Error("bad status");
      let data = {};
      try { data = await res.json(); } catch { /* health body is optional */ }
      const st = String(data?.status ?? "ok").toLowerCase();
      if (["error", "down", "unhealthy", "offline", "fail", "failed"].includes(st)) throw new Error("unhealthy");
      const rc = data?.records ?? data?.record_count ?? data?.dataset_size ?? data?.dataset_records ?? data?.num_records ?? data?.rows;
      setConn("online", Number.isFinite(Number(rc)) && rc !== null ? Number(rc) : undefined);
    } catch {
      setConn("offline");
    }
  }

  /* ---------- Safe rich-text rendering (textContent only) ---------- */
  function renderRich(container, text) {
    const lines = String(text).replace(/\r/g, "").split("\n");
    let list = null, listType = null, para = [];
    const flushPara = () => { if (para.length) { container.appendChild(el("p", null, para.join("\n"))); para = []; } };
    const flushList = () => { list = null; listType = null; };
    for (const raw of lines) {
      const line = raw.trim();
      if (!line) { flushPara(); flushList(); continue; }
      const num = line.match(/^\d+[.)]\s+(.*)$/);
      const bul = line.match(/^[-*\u2022]\s+(.*)$/);
      if (num || bul) {
        flushPara();
        const type = num ? "ol" : "ul";
        if (listType !== type) { list = el(type); container.appendChild(list); listType = type; }
        list.appendChild(el("li", null, (num || bul)[1]));
      } else { flushList(); para.push(line); }
    }
    flushPara();
  }

  /* ---------- Message rendering ---------- */
  function scrollToEnd() {
    log.scrollTo({ top: log.scrollHeight, behavior: reduceMotion() ? "auto" : "smooth" });
  }

  function addWelcome() {
    const m = el("div", "msg msg--bot");
    m.appendChild(logoAvatar());
    const col = el("div", "col");
    col.appendChild(el("div", "who", "NEXUS"));
    col.appendChild(el("div", "bubble", "Tell me what isn't working, and I'll look for a relevant answer in the support knowledge base."));
    m.appendChild(col);
    log.appendChild(m);
  }

  function addUser(text) {
    const m = el("div", "msg msg--user");
    const col = el("div", "col");
    col.appendChild(el("div", "who", "You"));
    col.appendChild(el("div", "bubble", text));
    m.appendChild(col);
    log.appendChild(m);
    state.hasChat = true;
    clearBtn.disabled = false;
    scrollToEnd();
  }

  function addTyping() {
    const m = el("div", "msg msg--bot");
    m.appendChild(logoAvatar());
    const col = el("div", "col");
    const b = el("div", "bubble");
    const t = el("div", "typing");
    t.append(el("i"), el("i"), el("i"), el("span", null, "Searching knowledge base…"));
    b.appendChild(t);
    col.appendChild(b);
    m.appendChild(col);
    log.appendChild(m);
    scrollToEnd();
    return m;
  }

  function feedbackBar({ allowRating }) {
    const id = ++state.seq;
    const bar = el("div", "fb");
    const note = el("span", "fb-note");
    note.setAttribute("role", "status");

    if (allowRating) {
      const mk = (label, value, down) => {
        const b = el("button", "ghost");
        b.type = "button";
        b.setAttribute("aria-pressed", "false");
        const ic = icon("i-thumb");
        if (down) ic.classList.add("down");
        b.append(ic, document.createTextNode(label));
        b.addEventListener("click", () => {
          state.feedback.push({ id, value });          // session-only; the question text is never stored
          bar.querySelectorAll("button[aria-pressed]").forEach(x => { x.disabled = true; });
          b.setAttribute("aria-pressed", "true");
          note.textContent = "Thanks. Your feedback is saved for this session only.";
        });
        return b;
      };
      bar.append(mk("Helpful", "helpful", false), mk("Not helpful", "not_helpful", true));
    }
    const another = el("button", "ghost");
    another.type = "button";
    another.append(icon("i-pencil"), document.createTextNode("Try another question"));
    another.addEventListener("click", () => { input.focus(); input.select(); });
    bar.append(another, note);
    return bar;
  }

  function detailsPanel(r) {
    const d = el("details", "details");
    const s = el("summary");
    s.append(el("span", null, "How this answer was matched"), icon("i-chevron"));
    d.appendChild(s);
    const body = el("div", "dbody");

    const q = el("div");
    q.append(el("div", "dlabel", "Matched dataset question"));
    q.append(el("div", "dq", r.matchedQuestion ? "\u201C" + r.matchedQuestion + "\u201D" : "Not provided by the API"));
    body.appendChild(q);

    const row = el("div", "drow");
    const sc = el("div");
    sc.append(el("div", "dlabel", "Cosine similarity"));
    sc.append(el("div", "score", Number.isFinite(r.score) ? r.score.toFixed(2) : "n/a"));
    const st = el("div");
    st.append(el("div", "dlabel", "Match status"));
    st.append(el("span", r.isMatch ? "pill pill--ok" : "pill pill--no", r.isMatch ? "Accepted" : "No confident match"));
    row.append(sc, st);
    body.appendChild(row);

    const note = el("div", "dnote");
    note.append(
      el("p", null, "This answer was retrieved from the existing knowledge base. It was not generated by an AI model."),
      el("p", null, "Cosine similarity shows how close your wording is to a stored question. It is not the probability that the answer is correct.")
    );
    body.appendChild(note);
    d.appendChild(body);
    return d;
  }

  function addBot(r) {
    const m = el("div", "msg msg--bot");
    m.appendChild(logoAvatar());
    const col = el("div", "col");
    col.appendChild(el("div", "who", "NEXUS"));
    const b = el("div", "bubble");

    if (r.isMatch && r.answer) {
      renderRich(b, r.answer);
      col.appendChild(b);
      col.appendChild(detailsPanel(r));
      col.appendChild(feedbackBar({ allowRating: true }));
    } else {
      const f = el("div", "fallback");
      f.appendChild(icon("i-search"));
      const t = el("div");
      const titles = { ambiguous: "Several issues look similar", clarify: "Can you clarify?", empty: "Nothing to search yet" };
      t.append(el("strong", null, titles[r.status] || "No confident match"), el("span", null, r.notice || FALLBACK_TEXT));
      f.appendChild(t);
      b.appendChild(f);
      col.appendChild(b);
      if (Number.isFinite(r.score) || r.matchedQuestion) col.appendChild(detailsPanel({ ...r, isMatch: false }));
      col.appendChild(feedbackBar({ allowRating: false }));
    }
    m.appendChild(col);
    log.appendChild(m);
    scrollToEnd();
  }

  const ERRORS = {
    network: "NEXUS can't reach the support service right now. Check your connection and try again.",
    timeout: "The support service took too long to respond. Please try again in a moment.",
    server: "The support service ran into a problem. Please try again shortly.",
    malformed: "The support service sent a response NEXUS couldn't read. Please try again."
  };

  function addError(kind, retryText) {
    const m = el("div", "msg msg--bot");
    m.appendChild(logoAvatar());
    const col = el("div", "col");
    col.appendChild(el("div", "who", "NEXUS"));
    const b = el("div", "bubble err");
    const f = el("div", "fallback");
    f.appendChild(icon("i-alert"));
    const t = el("div");
    t.append(el("strong", null, "Something went wrong"), el("span", null, ERRORS[kind] || ERRORS.server));
    f.appendChild(t);
    b.appendChild(f);
    const rb = el("button", "inline-btn");
    rb.type = "button";
    rb.append(icon("i-refresh"), document.createTextNode("Try again"));
    rb.addEventListener("click", () => { m.remove(); sendMessage(retryText, { retry: true }); });
    b.appendChild(rb);
    col.appendChild(b);
    m.appendChild(col);
    log.appendChild(m);
    scrollToEnd();
  }

  /* ---------- Backend call ---------- */
  class NexusError extends Error { constructor(kind) { super(kind); this.kind = kind; } }

  function normalizeChat(data) {
    if (!data || typeof data !== "object" || Array.isArray(data)) throw new NexusError("malformed");
    const pick = (...keys) => { for (const k of keys) if (data[k] !== undefined && data[k] !== null) return data[k]; };

    const rawAnswer = pick("answer", "response", "reply", "message");
    const answer = typeof rawAnswer === "string" ? rawAnswer.trim() : "";
    const mq = pick("matched_question", "matchedQuestion", "matched_query", "question");
    const matchedQuestion = typeof mq === "string" ? mq.trim() : "";
    const rawScore = pick("similarity", "score", "cosine_similarity", "cosineSimilarity", "similarity_score");
    const score = rawScore === undefined ? NaN : Number(rawScore);

    let flag = pick("is_match", "isMatch", "accepted", "confident", "found");
    if (flag === undefined && typeof data.matched === "boolean") flag = data.matched;
    if (flag === undefined && typeof data.status === "string") {
      const s = data.status.toLowerCase();
      if (["no_match", "no-match", "unmatched", "fallback", "rejected"].includes(s)) flag = false;
      if (["matched", "accepted", "match"].includes(s)) flag = true;
    }
    const isMatch = flag === undefined ? answer.length > 0 : Boolean(flag);

    if (rawAnswer === undefined && flag === undefined) throw new NexusError("malformed");
    const rn = pick("notice");
    const notice = typeof rn === "string" ? rn.trim() : "";
    const status = typeof data.status === "string" ? data.status.toLowerCase() : "";
    return { answer, matchedQuestion, score, isMatch, notice, status };
  }

  async function askBackend(text) {
    let res;
    try {
      res = await fetchWithTimeout(CONFIG.API_BASE_URL + CONFIG.CHAT_PATH, {
        method: "POST",
        headers: { "Content-Type": "application/json", Accept: "application/json" },
        body: JSON.stringify({ [CONFIG.REQUEST_FIELD]: text })
      }, CONFIG.CHAT_TIMEOUT_MS);
    } catch (e) {
      throw new NexusError(e && e.name === "AbortError" ? "timeout" : "network");
    }
    if (!res.ok) throw new NexusError("server");
    let data;
    try { data = await res.json(); } catch { throw new NexusError("malformed"); }
    return normalizeChat(data);
  }

  /* ---------- Send flow ---------- */
  function setBusy(v) {
    state.busy = v;
    form.setAttribute("aria-busy", String(v));
    updateSub();
    syncSend();
  }

  async function sendMessage(raw, opts = {}) {
    const text = String(raw || "").trim().slice(0, CONFIG.MAX_CHARS);
    if (!text) return;
    if (state.busy) { input.value = text; autosize(); syncSend(); input.focus(); return; }

    if (!opts.retry) addUser(text);
    input.value = "";
    autosize();
    setBusy(true);
    const typing = addTyping();

    try {
      const result = await askBackend(text);
      typing.remove();
      if (state.conn !== "online") setConn("online");
      addBot(result);
    } catch (err) {
      typing.remove();
      const kind = err instanceof NexusError ? err.kind : "network";
      addError(kind, text);
      if (kind === "network" || kind === "timeout") { setConn("offline"); }
    } finally {
      setBusy(false);
      input.focus({ preventScroll: true });
    }
  }

  /* ---------- Composer behaviour ---------- */
  function autosize() {
    input.style.height = "auto";
    input.style.height = Math.min(input.scrollHeight, 160) + "px";
  }
  function syncSend() {
    const len = input.value.length;
    sendBtn.disabled = state.busy || input.value.trim().length === 0;
    countEl.textContent = len + " / " + CONFIG.MAX_CHARS;
    countEl.dataset.warn = String(len > CONFIG.MAX_CHARS * 0.92);
  }

  input.addEventListener("input", () => { autosize(); syncSend(); });
  input.addEventListener("keydown", (e) => {
    if (e.key === "Enter" && !e.shiftKey && !e.isComposing) {
      e.preventDefault();
      if (!sendBtn.disabled) sendMessage(input.value);
    }
  });
  form.addEventListener("submit", (e) => {
    e.preventDefault();
    if (!sendBtn.disabled) sendMessage(input.value);
  });

  clearBtn.addEventListener("click", () => {
    if (state.busy) return;
    log.replaceChildren();
    state.hasChat = false;
    state.feedback = [];
    clearBtn.disabled = true;
    addWelcome();
    input.focus();
  });

  $("#retryConn").addEventListener("click", checkHealth);

  /* ---------- Build quick cards + chips ---------- */
  function startFromUI(query) {
    $("#chat").scrollIntoView({ behavior: reduceMotion() ? "auto" : "smooth", block: "start" });
    sendMessage(query);
  }

  const grid = $("#quickGrid");
  TOPICS.forEach((t) => {
    const card = el("article", "qcard");
    const ic = el("div", "qicon");
    ic.appendChild(icon(t.icon));
    const main = el("button", "qmain", t.title);
    main.type = "button";
    const descId = "qd-" + t.title.toLowerCase();
    main.setAttribute("aria-describedby", descId);
    main.addEventListener("click", () => startFromUI(t.main));
    const desc = el("p", "qdesc", t.desc);
    desc.id = descId;

    const ul = el("ul", "qlist");
    t.issues.forEach((q) => {
      const li = el("li");
      const b = el("button", "qissue");
      b.type = "button";
      b.append(el("span", null, q), icon("i-send"));
      b.addEventListener("click", () => startFromUI(q));
      li.appendChild(b);
      ul.appendChild(li);
    });
    card.append(ic, main, desc, ul);
    grid.appendChild(card);
  });

  const chips = $("#chips");
  SUGGESTIONS.forEach((q) => {
    const c = el("button", "chip", q);
    c.type = "button";
    c.addEventListener("click", () => sendMessage(q));
    chips.appendChild(c);
  });

  /* ---------- Init ---------- */
  addWelcome();
  syncSend();
  setConn("connecting");
  checkHealth();
  setInterval(() => { if (!document.hidden && !state.busy) checkHealth(); }, CONFIG.HEALTH_INTERVAL_MS);
  document.addEventListener("visibilitychange", () => { if (!document.hidden) checkHealth(); });
})();
</script>
</body>
</html>


In [ ]:
# Retrieval used by the NEXUS interface. Same model, thresholds, and normalization as chatbot().
MIN_SCORE = 0.40
MIN_MARGIN = 0.06
MAX_CHARS = 1200

NOTICE_NO_MATCH = (
    "I couldn't find a sufficiently relevant answer in the current knowledge base. "
    "Try including your device, operating system, and exact symptom."
)
NOTICE_AMBIGUOUS = (
    "A few issues sound similar, so I don't want to guess. "
    "Please add one more detail about the device and the exact symptom."
)
NOTICE_CLARIFY = (
    "Can you clarify the internet problem: is the connection missing, "
    "very slow, or connected but websites will not load?"
)

def _result(status, answer=None, notice=None, idx=None, score=None, margin=None):
    return {
        "status": status,              # ok | no_match | ambiguous | clarify | empty
        "is_match": status == "ok",
        "answer": answer,
        "notice": notice,
        "matched_question": issue_texts[idx] if idx is not None else None,
        "similarity": round(score, 4) if score is not None else None,
        "margin": round(margin, 4) if margin is not None else None,
    }

def retrieve(user_question):
    raw = str(user_question or "").strip()[:MAX_CHARS]
    if not raw:
        return _result("empty", notice="Please enter a question.")

    normalized = normalize(raw)
    if not normalized:
        return _result("no_match", notice=NOTICE_NO_MATCH)

    low = raw.lower()
    vague_internet = (
        re.search(r"\binternet\b", low)
        and re.search(r"not working|doesn't work|does not work|down", low)
        and not re.search(r"slow|wifi|wi-fi|ethernet|router|connected|browser|website|phone|laptop", low)
    )
    if vague_internet:
        return _result("clarify", notice=NOTICE_CLARIFY)

    scores = cosine_similarity(vectorizer.transform([normalized]), issue_vectors).ravel()
    best = int(np.argmax(scores))
    best_score = float(scores[best])
    ranked = np.argsort(scores)
    second = float(scores[ranked[-2]]) if len(ranked) > 1 else 0.0
    margin = best_score - second

    if best_score < MIN_SCORE:
        return _result("no_match", notice=NOTICE_NO_MATCH, idx=best, score=best_score, margin=margin)
    if margin < MIN_MARGIN:
        return _result("ambiguous", notice=NOTICE_AMBIGUOUS, idx=best, score=best_score, margin=margin)
    return _result("ok", answer=issue_answers[best], idx=best, score=best_score, margin=margin)

print("Retrieval ready. Test:", retrieve("My PC won't start")["status"])


In [ ]:
import gradio as gr
from pathlib import Path
from starlette.routing import Route
from starlette.responses import HTMLResponse, JSONResponse
from starlette.concurrency import run_in_threadpool

SHARE = True   # True = public gradio.live link (free, expires after ~1 week)

# Re-running this cell: stop the previous server first
try:
    demo.close()
except Exception:
    pass

PAGE = Path("index.html").read_text(encoding="utf-8")

async def nexus_home(request):
    return HTMLResponse(PAGE)

async def nexus_health(request):
    return JSONResponse({"status": "ok", "records": len(issue_texts), "qa_pairs": len(rows)})

async def nexus_chat(request):
    try:
        body = await request.json()
    except Exception:
        body = {}
    message = body.get("message", "") if isinstance(body, dict) else ""
    return JSONResponse(await run_in_threadpool(retrieve, message))

# Gradio needs at least one Blocks app to start its server; the NEXUS page replaces its home page.
with gr.Blocks(title="NEXUS IT Support Assistant") as demo:
    gr.Markdown("NEXUS is running.")

app, local_url, share_url = demo.launch(share=SHARE, prevent_thread_lock=True, quiet=True)

# Put NEXUS routes ahead of Gradio's own routes
for route in reversed([
    Route("/", nexus_home, methods=["GET"]),
    Route("/api/health", nexus_health, methods=["GET"]),
    Route("/api/chat", nexus_chat, methods=["POST"]),
]):
    app.router.routes.insert(0, route)

print("Local link :", local_url)
print("Public link:", share_url or "not created (check your internet connection or set SHARE = True)")
